In [1]:
!pip install -q ultralytics scikit-learn grad-cam

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 41.3/41.3 kB 1.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 53.9 MB/s eta 0:00:0000:0100:01
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 54.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 73.7 MB/s eta 0:00:00:00:010:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26.2.0 requires numba-cuda<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
distributed-ucxx-cu12 0.48.0 requires numba-cuda[cu12]<0.23.0,>=0.22.1, but you have numba-cuda 0.30.2 which is incompatible.
cuml-cu12 26.2.0 requires

In [2]:
import gc
import random
import shutil
import warnings

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from PIL import Image
from ultralytics import YOLO

from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_curve,
    precision_recall_fscore_support,
    roc_auc_score,
    roc_curve,
)

from sklearn.preprocessing import label_binarize

warnings.filterwarnings("ignore")

print("PyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
PyTorch version: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
SEEDS = [0, 42, 123]

EPOCHS = 50
IMAGE_SIZE = 224
BATCH_SIZE = 32
WORKERS = 2

DEVICE = 0 if torch.cuda.is_available() else "cpu"

OUTPUT_ROOT = Path("/kaggle/working/brain_tumor_yolo11")

DATASET_DIR = OUTPUT_ROOT / "fixed_dataset"
RUNS_DIR = OUTPUT_ROOT / "runs"
RESULTS_DIR = OUTPUT_ROOT / "results"
GRADCAM_DIR = OUTPUT_ROOT / "gradcam"

for folder in [
    OUTPUT_ROOT,
    RUNS_DIR,
    RESULTS_DIR,
    GRADCAM_DIR,
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Device:", DEVICE)
print("Output folder:", OUTPUT_ROOT)

Device: 0
Output folder: /kaggle/working/brain_tumor_yolo11


In [4]:
INPUT_ROOT = Path("/kaggle/input")

training_candidates = [
    path
    for path in INPUT_ROOT.rglob("Training")
    if path.is_dir()
]

testing_candidates = [
    path
    for path in INPUT_ROOT.rglob("Testing")
    if path.is_dir()
]

print("Training folders found:")
for path in training_candidates:
    print(path)

print("\nTesting folders found:")
for path in testing_candidates:
    print(path)

if not training_candidates:
    raise FileNotFoundError(
        "No Training folder found. Add the Brain Tumor MRI Dataset "
        "to the Kaggle notebook input."
    )

if not testing_candidates:
    raise FileNotFoundError(
        "No Testing folder found. Add the Brain Tumor MRI Dataset "
        "to the Kaggle notebook input."
    )

TRAIN_SOURCE = training_candidates[0]
TEST_SOURCE = testing_candidates[0]

print("\nSelected Training folder:", TRAIN_SOURCE)
print("Selected Testing folder :", TEST_SOURCE)

Training folders found:
/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training

Testing folders found:
/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing

Selected Training folder: /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Training
Selected Testing folder : /kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset/Testing


In [5]:
VALID_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".bmp",
    ".tif",
    ".tiff",
    ".webp",
}


def get_image_files(folder):
    return sorted(
        path
        for path in Path(folder).rglob("*")
        if path.is_file()
        and path.suffix.lower() in VALID_EXTENSIONS
    )


def get_class_counts(split_folder):
    counts = {}

    for class_folder in sorted(Path(split_folder).iterdir()):
        if class_folder.is_dir():
            counts[class_folder.name] = len(
                get_image_files(class_folder)
            )

    return counts


train_source_counts = get_class_counts(TRAIN_SOURCE)
test_source_counts = get_class_counts(TEST_SOURCE)

print("Training source counts:")
for class_name, count in train_source_counts.items():
    print(f"{class_name}: {count}")

print("\nTesting source counts:")
for class_name, count in test_source_counts.items():
    print(f"{class_name}: {count}")

print("\nTotal training images:", sum(train_source_counts.values()))
print("Total testing images :", sum(test_source_counts.values()))

train_classes = set(train_source_counts)
test_classes = set(test_source_counts)

if train_classes != test_classes:
    raise ValueError(
        f"Class mismatch between Training and Testing.\n"
        f"Training classes: {sorted(train_classes)}\n"
        f"Testing classes : {sorted(test_classes)}"
    )

CLASS_NAMES = sorted(train_classes)

print("\nClasses:", CLASS_NAMES)
print("Number of classes:", len(CLASS_NAMES))

Training source counts:
glioma: 1400
meningioma: 1400
notumor: 1400
pituitary: 1400

Testing source counts:
glioma: 400
meningioma: 400
notumor: 400
pituitary: 400

Total training images: 5600
Total testing images : 1600

Classes: ['glioma', 'meningioma', 'notumor', 'pituitary']
Number of classes: 4


In [6]:
if DATASET_DIR.exists():
    shutil.rmtree(DATASET_DIR)

for split_name in ["train", "val", "test"]:
    (DATASET_DIR / split_name).mkdir(
        parents=True,
        exist_ok=True,
    )

SPLIT_SEED = 123
VALIDATION_RATIO = 0.20

random.seed(SPLIT_SEED)


def copy_files(file_paths, destination):
    destination.mkdir(
        parents=True,
        exist_ok=True,
    )

    for source_path in file_paths:
        shutil.copy2(
            source_path,
            destination / source_path.name,
        )


training_classes = sorted([
    path.name
    for path in TRAIN_SOURCE.iterdir()
    if path.is_dir()
])

for class_name in training_classes:
    class_files = get_image_files(
        TRAIN_SOURCE / class_name
    )

    random.shuffle(class_files)

    number_validation = int(
        len(class_files) * VALIDATION_RATIO
    )

    validation_files = class_files[:number_validation]
    training_files = class_files[number_validation:]

    copy_files(
        training_files,
        DATASET_DIR / "train" / class_name,
    )

    copy_files(
        validation_files,
        DATASET_DIR / "val" / class_name,
    )


testing_classes = sorted([
    path.name
    for path in TEST_SOURCE.iterdir()
    if path.is_dir()
])

for class_name in testing_classes:
    testing_files = get_image_files(
        TEST_SOURCE / class_name
    )

    copy_files(
        testing_files,
        DATASET_DIR / "test" / class_name,
    )

print("Fixed dataset created at:", DATASET_DIR)

Fixed dataset created at: /kaggle/working/brain_tumor_yolo11/fixed_dataset


In [7]:
def count_split(split_name):
    split_path = DATASET_DIR / split_name
    counts = {}

    for class_folder in sorted(split_path.iterdir()):
        if class_folder.is_dir():
            counts[class_folder.name] = len(
                get_image_files(class_folder)
            )

    return counts


split_counts = {}

for split_name in ["train", "val", "test"]:
    counts = count_split(split_name)
    split_counts[split_name] = counts

    print(f"\n{split_name.upper()}:")
    print(counts)
    print("Total:", sum(counts.values()))


TRAIN:
{'glioma': 1120, 'meningioma': 1120, 'notumor': 1120, 'pituitary': 1120}
Total: 4480

VAL:
{'glioma': 280, 'meningioma': 280, 'notumor': 280, 'pituitary': 280}
Total: 1120

TEST:
{'glioma': 400, 'meningioma': 400, 'notumor': 400, 'pituitary': 400}
Total: 1600


In [8]:
EXPECTED_TOTALS = {
    "train": 4480,
    "val": 1120,
    "test": 1600,
}

for split_name, expected_total in EXPECTED_TOTALS.items():
    actual_total = sum(split_counts[split_name].values())

    assert actual_total == expected_total, (
        f"{split_name}: expected {expected_total}, "
        f"but found {actual_total}"
    )


manifest_records = []

for split_name in ["train", "val", "test"]:
    split_path = DATASET_DIR / split_name

    for class_folder in sorted(split_path.iterdir()):
        if not class_folder.is_dir():
            continue

        for image_path in get_image_files(class_folder):
            manifest_records.append(
                {
                    "split": split_name,
                    "class_name": class_folder.name,
                    "filename": image_path.name,
                    "filepath": str(image_path),
                }
            )


manifest_df = pd.DataFrame(manifest_records)

manifest_path = RESULTS_DIR / "fixed_dataset_manifest.csv"

manifest_df.to_csv(
    manifest_path,
    index=False,
)

display(
    manifest_df
    .groupby(["split", "class_name"])
    .size()
    .unstack(fill_value=0)
)

print("Dataset verified.")
print("Manifest saved at:", manifest_path)

class_name,glioma,meningioma,notumor,pituitary
split,,,,
test,400,400,400,400
train,1120,1120,1120,1120
val,280,280,280,280


Dataset verified.
Manifest saved at: /kaggle/working/brain_tumor_yolo11/results/fixed_dataset_manifest.csv


In [9]:
def set_all_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def clear_memory():
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()


print("Helper functions ready.")

Helper functions ready.


In [10]:
def train_yolo11(seed, augmentation_mode):
    set_all_seeds(seed)
    clear_memory()

    if augmentation_mode == "default":
        run_prefix = "yolo11n_default"

    elif augmentation_mode == "mri":
        run_prefix = "yolo11n_mri_aug"

    else:
        raise ValueError(
            "augmentation_mode must be 'default' or 'mri'"
        )

    run_name = f"{run_prefix}_seed{seed}"
    run_directory = RUNS_DIR / run_name

    best_checkpoint = (
        run_directory
        / "weights"
        / "best.pt"
    )

    last_checkpoint = (
        run_directory
        / "weights"
        / "last.pt"
    )

    if best_checkpoint.exists():
        print(f"Completed checkpoint already exists: {run_name}")
        return str(best_checkpoint)

    print("\n" + "=" * 90)
    print("Run:", run_name)
    print("Seed:", seed)
    print("Augmentation:", augmentation_mode)
    print("=" * 90)

    if last_checkpoint.exists():
        print("Resuming from:", last_checkpoint)

        model = YOLO(str(last_checkpoint))
        model.train(resume=True)

    else:
        if run_directory.exists():
            shutil.rmtree(run_directory)

        model = YOLO("yolo11n-cls.pt")

        common_arguments = {
            "data": str(DATASET_DIR),
            "epochs": EPOCHS,
            "imgsz": IMAGE_SIZE,
            "batch": BATCH_SIZE,
            "workers": WORKERS,
            "seed": seed,
            "deterministic": True,
            "pretrained": True,
            "optimizer": "auto",
            "device": DEVICE,
            "save": True,
            "save_period": 5,
            "plots": True,
            "project": str(RUNS_DIR),
            "name": run_name,
            "exist_ok": False,
        }

        if augmentation_mode == "default":
            model.train(**common_arguments)

        else:
            model.train(
                **common_arguments,
                auto_augment=None,
                hsv_h=0.0,
                hsv_s=0.0,
                hsv_v=0.10,
                degrees=5.0,
                translate=0.05,
                scale=0.10,
                fliplr=0.5,
                flipud=0.0,
                erasing=0.10,
                mixup=0.0,
                cutmix=0.0,
            )

    if not best_checkpoint.exists():
        raise FileNotFoundError(
            f"Training ended but best.pt was not found: "
            f"{best_checkpoint}"
        )

    print("Best checkpoint:", best_checkpoint)

    clear_memory()

    return str(best_checkpoint)


print("YOLO11 training function ready.")

YOLO11 training function ready.


In [ ]:
yolo11_default_seed0 = train_yolo11(
    seed=0,
    augmentation_mode="default",
)


Run: yolo11n_default_seed0
Seed: 0
Augmentation: default
Ultralytics 8.4.75 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14912MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=32, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/brain_tumor_yolo11/fixed_dataset, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=224, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo11n-cls.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolo11n_default_seed0, n

In [ ]:
yolo11_default_seed42 = train_yolo11(
    seed=42,
    augmentation_mode="default",
)

In [ ]:
yolo11_default_seed123 = train_yolo11(
    seed=123,
    augmentation_mode="default",
)

In [ ]:
yolo11_mri_seed0 = train_yolo11(
    seed=0,
    augmentation_mode="mri",
)

In [ ]:
yolo11_mri_seed42 = train_yolo11(
    seed=42,
    augmentation_mode="mri",
)

In [ ]:
yolo11_mri_seed123 = train_yolo11(
    seed=123,
    augmentation_mode="mri",
)

In [ ]:
all_checkpoints = {
    "default_seed0": Path(yolo11_default_seed0),
    "default_seed42": Path(yolo11_default_seed42),
    "default_seed123": Path(yolo11_default_seed123),
    "mri_seed0": Path(yolo11_mri_seed0),
    "mri_seed42": Path(yolo11_mri_seed42),
    "mri_seed123": Path(yolo11_mri_seed123),
}

for run_name, checkpoint_path in all_checkpoints.items():
    if not checkpoint_path.exists():
        raise FileNotFoundError(
            f"Missing checkpoint for {run_name}: {checkpoint_path}"
        )

    print(f"✅ {run_name}: {checkpoint_path}")

print("\nAll six YOLO11 checkpoints are available.")

In [ ]:
def evaluate_classifier(checkpoint_path, split_name, run_name):
    clear_memory()

    model = YOLO(str(checkpoint_path))
    split_path = DATASET_DIR / split_name

    image_paths = sorted([
        str(path)
        for path in split_path.rglob("*")
        if path.is_file()
        and path.suffix.lower() in VALID_EXTENSIONS
    ])

    if not image_paths:
        raise FileNotFoundError(
            f"No images found recursively inside: {split_path}"
        )

    print(f"Found {len(image_paths)} images in {split_name}")

    results = model.predict(
        source=image_paths,
        imgsz=IMAGE_SIZE,
        batch=BATCH_SIZE,
        device=DEVICE,
        verbose=False,
        stream=False,
    )

    y_true = []
    y_pred = []
    y_prob = []
    processed_paths = []

    class_to_index = {
        class_name: index
        for index, class_name in enumerate(CLASS_NAMES)
    }

    for result in results:
        image_path = Path(result.path)
        true_class_name = image_path.parent.name

        probabilities = (
            result.probs.data
            .detach()
            .cpu()
            .numpy()
        )

        predicted_index = int(np.argmax(probabilities))
        true_index = class_to_index[true_class_name]

        y_true.append(true_index)
        y_pred.append(predicted_index)
        y_prob.append(probabilities)
        processed_paths.append(str(image_path))

    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    y_prob = np.asarray(y_prob)

    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true,
        y_pred,
        average="macro",
        zero_division=0,
    )

    accuracy = accuracy_score(y_true, y_pred)
    balanced_accuracy = balanced_accuracy_score(y_true, y_pred)
    mcc = matthews_corrcoef(y_true, y_pred)

    y_true_binary = label_binarize(
        y_true,
        classes=np.arange(len(CLASS_NAMES)),
    )

    try:
        macro_roc_auc = roc_auc_score(
            y_true_binary,
            y_prob,
            average="macro",
            multi_class="ovr",
        )
    except ValueError:
        macro_roc_auc = np.nan

    try:
        macro_average_precision = average_precision_score(
            y_true_binary,
            y_prob,
            average="macro",
        )
    except ValueError:
        macro_average_precision = np.nan

    metrics = {
        "run_name": run_name,
        "split": split_name,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_precision": precision,
        "macro_recall": recall,
        "macro_f1": f1,
        "mcc": mcc,
        "macro_roc_auc": macro_roc_auc,
        "macro_average_precision": macro_average_precision,
    }

    predictions_df = pd.DataFrame({
        "image_path": processed_paths,
        "true_index": y_true,
        "predicted_index": y_pred,
        "true_class": [CLASS_NAMES[i] for i in y_true],
        "predicted_class": [CLASS_NAMES[i] for i in y_pred],
    })

    for class_index, class_name in enumerate(CLASS_NAMES):
        predictions_df[f"prob_{class_name}"] = y_prob[:, class_index]

    print(f"\n{run_name} | {split_name}")
    print(f"Accuracy          : {accuracy:.4f}")
    print(f"Balanced Accuracy : {balanced_accuracy:.4f}")
    print(f"Macro Precision   : {precision:.4f}")
    print(f"Macro Recall      : {recall:.4f}")
    print(f"Macro F1          : {f1:.4f}")
    print(f"MCC               : {mcc:.4f}")
    print(f"Macro ROC-AUC     : {macro_roc_auc:.4f}")
    print(f"Macro AP          : {macro_average_precision:.4f}")

    clear_memory()

    return metrics, predictions_df


print("YOLO11 evaluation function ready.")

In [ ]:
CLASS_NAMES = sorted([
    folder.name
    for folder in (DATASET_DIR / "train").iterdir()
    if folder.is_dir()
])

all_metrics = []
all_predictions = {}

for run_name, checkpoint_path in all_checkpoints.items():
    for split_name in ["val", "test"]:
        print("\n" + "=" * 90)

        metrics, predictions_df = evaluate_classifier(
            checkpoint_path=checkpoint_path,
            split_name=split_name,
            run_name=run_name,
        )

        all_metrics.append(metrics)

        prediction_key = f"{run_name}_{split_name}"
        all_predictions[prediction_key] = predictions_df

        predictions_df.to_csv(
            RESULTS_DIR / f"{prediction_key}_predictions.csv",
            index=False,
        )

metrics_df = pd.DataFrame(all_metrics)

metrics_path = RESULTS_DIR / "all_model_metrics.csv"

metrics_df.to_csv(
    metrics_path,
    index=False,
)

display(
    metrics_df.sort_values(
        by=["split", "macro_f1"],
        ascending=[True, False],
    ).reset_index(drop=True)
)

print("\nMetrics saved at:", metrics_path)
print("Prediction files saved in:", RESULTS_DIR)

In [ ]:
metrics_df["augmentation"] = metrics_df["run_name"].apply(
    lambda name: "default" if name.startswith("default") else "mri"
)

metric_columns = [
    "accuracy",
    "balanced_accuracy",
    "macro_precision",
    "macro_recall",
    "macro_f1",
    "mcc",
    "macro_roc_auc",
    "macro_average_precision",
]

summary_rows = []

for split_name in ["val", "test"]:
    for augmentation_mode in ["default", "mri"]:
        subset = metrics_df[
            (metrics_df["split"] == split_name)
            & (metrics_df["augmentation"] == augmentation_mode)
        ]

        row = {
            "split": split_name,
            "augmentation": augmentation_mode,
            "number_of_runs": len(subset),
        }

        for metric_name in metric_columns:
            metric_mean = subset[metric_name].mean()
            metric_std = subset[metric_name].std(ddof=1)

            row[f"{metric_name}_mean"] = metric_mean
            row[f"{metric_name}_std"] = metric_std
            row[f"{metric_name}_mean_std"] = (
                f"{metric_mean:.4f} ± {metric_std:.4f}"
            )

        summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)

summary_path = RESULTS_DIR / "seed_aggregated_metrics.csv"

summary_df.to_csv(
    summary_path,
    index=False,
)

display_columns = [
    "split",
    "augmentation",
    "accuracy_mean_std",
    "balanced_accuracy_mean_std",
    "macro_precision_mean_std",
    "macro_recall_mean_std",
    "macro_f1_mean_std",
    "mcc_mean_std",
    "macro_roc_auc_mean_std",
    "macro_average_precision_mean_std",
]

display(summary_df[display_columns])

print("Aggregated YOLO11 results saved at:", summary_path)

In [ ]:
validation_results = metrics_df[
    metrics_df["split"] == "val"
].copy()

best_validation_row = validation_results.loc[
    validation_results["macro_f1"].idxmax()
]

BEST_RUN_NAME = best_validation_row["run_name"]
BEST_CHECKPOINT = all_checkpoints[BEST_RUN_NAME]

print("Best YOLO11 run selected using validation Macro-F1")
print("-" * 60)
print("Run name            :", BEST_RUN_NAME)
print("Checkpoint          :", BEST_CHECKPOINT)
print("Validation Accuracy :", f"{best_validation_row['accuracy']:.4f}")
print("Validation Macro-F1 :", f"{best_validation_row['macro_f1']:.4f}")
print("Validation MCC      :", f"{best_validation_row['mcc']:.4f}")
print("Validation ROC-AUC  :", f"{best_validation_row['macro_roc_auc']:.4f}")

best_model_summary = pd.DataFrame([
    {
        "best_run_name": BEST_RUN_NAME,
        "checkpoint": str(BEST_CHECKPOINT),
        "selection_split": "val",
        "selection_metric": "macro_f1",
        "validation_accuracy": best_validation_row["accuracy"],
        "validation_macro_f1": best_validation_row["macro_f1"],
        "validation_mcc": best_validation_row["mcc"],
        "validation_macro_roc_auc": best_validation_row["macro_roc_auc"],
    }
])

best_model_summary_path = (
    RESULTS_DIR / "best_model_selection.csv"
)

best_model_summary.to_csv(
    best_model_summary_path,
    index=False,
)

print("\nBest-model summary saved at:", best_model_summary_path)

In [ ]:
best_test_key = f"{BEST_RUN_NAME}_test"

best_test_predictions = all_predictions[
    best_test_key
].copy()

y_true_best = best_test_predictions[
    "true_index"
].to_numpy()

y_pred_best = best_test_predictions[
    "predicted_index"
].to_numpy()

confusion_matrix_values = confusion_matrix(
    y_true_best,
    y_pred_best,
)

confusion_matrix_df = pd.DataFrame(
    confusion_matrix_values,
    index=CLASS_NAMES,
    columns=CLASS_NAMES,
)

confusion_matrix_path = (
    RESULTS_DIR
    / f"{BEST_RUN_NAME}_test_confusion_matrix.csv"
)

confusion_matrix_df.to_csv(confusion_matrix_path)

plt.figure(figsize=(8, 6))

plt.imshow(
    confusion_matrix_values,
    interpolation="nearest",
)

plt.title(
    f"Confusion Matrix — {BEST_RUN_NAME}"
)

plt.colorbar()

tick_positions = np.arange(
    len(CLASS_NAMES)
)

plt.xticks(
    tick_positions,
    CLASS_NAMES,
    rotation=45,
    ha="right",
)

plt.yticks(
    tick_positions,
    CLASS_NAMES,
)

threshold = confusion_matrix_values.max() / 2

for row_index in range(
    confusion_matrix_values.shape[0]
):
    for column_index in range(
        confusion_matrix_values.shape[1]
    ):
        value = confusion_matrix_values[
            row_index,
            column_index,
        ]

        plt.text(
            column_index,
            row_index,
            str(value),
            horizontalalignment="center",
            verticalalignment="center",
            color="white" if value > threshold else "black",
        )

plt.ylabel("True Class")
plt.xlabel("Predicted Class")
plt.tight_layout()

confusion_matrix_figure_path = (
    RESULTS_DIR
    / f"{BEST_RUN_NAME}_test_confusion_matrix.png"
)

plt.savefig(
    confusion_matrix_figure_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

classification_report_dict = classification_report(
    y_true_best,
    y_pred_best,
    target_names=CLASS_NAMES,
    output_dict=True,
    zero_division=0,
)

classification_report_df = (
    pd.DataFrame(
        classification_report_dict
    )
    .transpose()
)

classification_report_path = (
    RESULTS_DIR
    / f"{BEST_RUN_NAME}_test_classification_report.csv"
)

classification_report_df.to_csv(
    classification_report_path
)

display(confusion_matrix_df)
display(classification_report_df)

print("Confusion matrix saved at:", confusion_matrix_path)
print("Confusion matrix image saved at:", confusion_matrix_figure_path)
print("Classification report saved at:", classification_report_path)

In [ ]:
probability_columns = [
    f"prob_{class_name}"
    for class_name in CLASS_NAMES
]

y_prob_best = best_test_predictions[
    probability_columns
].to_numpy()

y_true_best_binary = label_binarize(
    y_true_best,
    classes=np.arange(len(CLASS_NAMES)),
)

roc_auc_records = []
pr_auc_records = []


# ROC CURVES
plt.figure(figsize=(8, 6))

for class_index, class_name in enumerate(CLASS_NAMES):
    false_positive_rate, true_positive_rate, _ = roc_curve(
        y_true_best_binary[:, class_index],
        y_prob_best[:, class_index],
    )

    class_roc_auc = roc_auc_score(
        y_true_best_binary[:, class_index],
        y_prob_best[:, class_index],
    )

    roc_auc_records.append({
        "class_name": class_name,
        "roc_auc": class_roc_auc,
    })

    plt.plot(
        false_positive_rate,
        true_positive_rate,
        label=f"{class_name} (AUC = {class_roc_auc:.4f})",
    )

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--",
)

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title(f"ROC Curves — {BEST_RUN_NAME}")
plt.legend()
plt.grid(True)
plt.tight_layout()

roc_curve_path = (
    RESULTS_DIR
    / f"{BEST_RUN_NAME}_test_roc_curves.png"
)

plt.savefig(
    roc_curve_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()


# PRECISION-RECALL CURVES
plt.figure(figsize=(8, 6))

for class_index, class_name in enumerate(CLASS_NAMES):
    precision_values, recall_values, _ = precision_recall_curve(
        y_true_best_binary[:, class_index],
        y_prob_best[:, class_index],
    )

    class_average_precision = average_precision_score(
        y_true_best_binary[:, class_index],
        y_prob_best[:, class_index],
    )

    pr_auc_records.append({
        "class_name": class_name,
        "average_precision": class_average_precision,
    })

    plt.plot(
        recall_values,
        precision_values,
        label=(
            f"{class_name} "
            f"(AP = {class_average_precision:.4f})"
        ),
    )

plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title(
    f"Precision–Recall Curves — {BEST_RUN_NAME}"
)
plt.legend()
plt.grid(True)
plt.tight_layout()

pr_curve_path = (
    RESULTS_DIR
    / f"{BEST_RUN_NAME}_test_precision_recall_curves.png"
)

plt.savefig(
    pr_curve_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()


roc_auc_df = pd.DataFrame(roc_auc_records)
pr_auc_df = pd.DataFrame(pr_auc_records)

roc_auc_df.to_csv(
    RESULTS_DIR / f"{BEST_RUN_NAME}_per_class_roc_auc.csv",
    index=False,
)

pr_auc_df.to_csv(
    RESULTS_DIR / f"{BEST_RUN_NAME}_per_class_average_precision.csv",
    index=False,
)

display(roc_auc_df)
display(pr_auc_df)

print("ROC curve saved at:", roc_curve_path)
print("Precision–Recall curve saved at:", pr_curve_path)

In [ ]:
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.image import show_cam_on_image
from torchvision import transforms


clear_memory()
torch.set_grad_enabled(True)

gradcam_yolo = YOLO(str(BEST_CHECKPOINT))
gradcam_model = gradcam_yolo.model

gradcam_model.to(DEVICE)
gradcam_model.float()
gradcam_model.eval()

for parameter in gradcam_model.parameters():
    parameter.requires_grad_(True)


class YOLO11ClassificationWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, images):
        images = images.requires_grad_(True)

        output = self.model(images)

        if isinstance(output, tuple):
            output = output[0]

        if isinstance(output, list):
            output = output[0]

        return output


wrapped_gradcam_model = YOLO11ClassificationWrapper(
    gradcam_model
).to(DEVICE)

wrapped_gradcam_model.eval()


classification_head = gradcam_model.model[-1]

if hasattr(classification_head, "conv"):
    target_layer = classification_head.conv
else:
    target_layer = gradcam_model.model[-2]


gradcam_transform = transforms.Compose([
    transforms.Resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    ),
    transforms.ToTensor(),
])


cam = GradCAM(
    model=wrapped_gradcam_model,
    target_layers=[target_layer],
)


test_tensor = torch.randn(
    1,
    3,
    IMAGE_SIZE,
    IMAGE_SIZE,
    device=DEVICE,
    requires_grad=True,
)

test_output = wrapped_gradcam_model(test_tensor)

print("Best YOLO11 model:", BEST_RUN_NAME)
print("Checkpoint:", BEST_CHECKPOINT)
print("Output shape:", test_output.shape)
print("Output requires gradient:", test_output.requires_grad)
print("Grad-CAM target layer:", target_layer)

assert test_output.requires_grad, (
    "Model output does not support gradients."
)

del test_tensor, test_output
clear_memory()

print("YOLO11 Grad-CAM setup complete.")

In [ ]:
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget


GRADCAM_SAMPLES_PER_CLASS = 3

selected_gradcam_samples = []

for class_name in CLASS_NAMES:
    class_samples = best_test_predictions[
        best_test_predictions["true_class"] == class_name
    ].copy()

    correctly_classified = class_samples[
        class_samples["true_class"]
        == class_samples["predicted_class"]
    ]

    if len(correctly_classified) >= GRADCAM_SAMPLES_PER_CLASS:
        selected_samples = correctly_classified.sample(
            n=GRADCAM_SAMPLES_PER_CLASS,
            random_state=123,
        )
    else:
        selected_samples = class_samples.sample(
            n=min(
                GRADCAM_SAMPLES_PER_CLASS,
                len(class_samples),
            ),
            random_state=123,
        )

    selected_gradcam_samples.append(selected_samples)


selected_gradcam_df = pd.concat(
    selected_gradcam_samples,
    ignore_index=True,
)

gradcam_records = []


for sample_index, sample_row in selected_gradcam_df.iterrows():
    image_path = Path(sample_row["image_path"])

    true_class = sample_row["true_class"]
    predicted_class = sample_row["predicted_class"]
    predicted_index = int(sample_row["predicted_index"])

    pil_image = Image.open(image_path).convert("RGB")

    resized_image = pil_image.resize(
        (IMAGE_SIZE, IMAGE_SIZE)
    )

    rgb_image = (
        np.asarray(resized_image)
        .astype(np.float32)
        / 255.0
    )

    input_tensor = gradcam_transform(
        pil_image
    ).unsqueeze(0).to(DEVICE)

    input_tensor.requires_grad_(True)

    targets = [
        ClassifierOutputTarget(predicted_index)
    ]

    grayscale_cam = cam(
        input_tensor=input_tensor,
        targets=targets,
    )[0]

    gradcam_visualization = show_cam_on_image(
        rgb_image,
        grayscale_cam,
        use_rgb=True,
    )

    output_filename = (
        f"{sample_index + 1:02d}_"
        f"true_{true_class}_"
        f"pred_{predicted_class}.png"
    )

    output_path = GRADCAM_DIR / output_filename

    Image.fromarray(
        gradcam_visualization
    ).save(output_path)

    gradcam_records.append({
        "image_path": str(image_path),
        "true_class": true_class,
        "predicted_class": predicted_class,
        "gradcam_path": str(output_path),
    })


gradcam_records_df = pd.DataFrame(
    gradcam_records
)

gradcam_records_path = (
    RESULTS_DIR
    / "gradcam_visualization_records.csv"
)

gradcam_records_df.to_csv(
    gradcam_records_path,
    index=False,
)


number_of_samples = len(gradcam_records_df)

plt.figure(
    figsize=(15, 4 * number_of_samples)
)

for index, record in gradcam_records_df.iterrows():
    original_image = Image.open(
        record["image_path"]
    ).convert("RGB")

    gradcam_image = Image.open(
        record["gradcam_path"]
    ).convert("RGB")

    plt.subplot(
        number_of_samples,
        2,
        (index * 2) + 1,
    )

    plt.imshow(original_image)
    plt.title(
        f"Original\nTrue: {record['true_class']}"
    )
    plt.axis("off")

    plt.subplot(
        number_of_samples,
        2,
        (index * 2) + 2,
    )

    plt.imshow(gradcam_image)
    plt.title(
        f"Grad-CAM\nPredicted: {record['predicted_class']}"
    )
    plt.axis("off")


plt.tight_layout()

gradcam_grid_path = (
    GRADCAM_DIR
    / f"{BEST_RUN_NAME}_gradcam_grid.png"
)

plt.savefig(
    gradcam_grid_path,
    dpi=300,
    bbox_inches="tight",
)

plt.show()

display(gradcam_records_df)

print("Grad-CAM images saved in:", GRADCAM_DIR)
print("Grad-CAM grid saved at:", gradcam_grid_path)
print("Grad-CAM records saved at:", gradcam_records_path)

In [ ]:
final_summary = metrics_df[
    metrics_df["run_name"] == BEST_RUN_NAME
].copy()

final_summary = final_summary[
    [
        "run_name",
        "split",
        "accuracy",
        "balanced_accuracy",
        "macro_precision",
        "macro_recall",
        "macro_f1",
        "mcc",
        "macro_roc_auc",
        "macro_average_precision",
    ]
]

final_summary_path = (
    RESULTS_DIR
    / "final_best_model_results.csv"
)

final_summary.to_csv(
    final_summary_path,
    index=False,
)

display(final_summary)


archive_base_path = (
    Path("/kaggle/working")
    / "brain_tumor_yolo11_complete_results"
)

archive_path = shutil.make_archive(
    base_name=str(archive_base_path),
    format="zip",
    root_dir=str(OUTPUT_ROOT),
)

print("\nYOLO11 experiment completed successfully.")
print("Best model:", BEST_RUN_NAME)
print("Best checkpoint:", BEST_CHECKPOINT)
print("Final summary saved at:", final_summary_path)
print("Complete ZIP package created at:", archive_path)

In [ ]:
print("Best run:", BEST_RUN_NAME)
print("Best checkpoint exists:", Path(BEST_CHECKPOINT).exists())
print("Results folder exists:", RESULTS_DIR.exists())
print("Grad-CAM folder exists:", GRADCAM_DIR.exists())
print("ZIP exists:", Path(archive_path).exists())

print("\nGenerated result files:")
for file_path in sorted(RESULTS_DIR.rglob("*")):
    if file_path.is_file():
        print(file_path.name)

print("\nGenerated Grad-CAM files:")
for file_path in sorted(GRADCAM_DIR.rglob("*")):
    if file_path.is_file():
        print(file_path.name)

print("\nFinal selected-model results:")
display(final_summary)